In [27]:
# Cell 1: Setup + UNF (Unnormalized Form)
import sqlite3
import pandas as pd

# Use persistent database file
conn = sqlite3.connect("superstore_normalization.db")
cursor = conn.cursor()

print("=== UNF: Unnormalized Form ===")
cursor.execute('''
CREATE TABLE IF NOT EXISTS orders_unf (
    order_id INTEGER,
    order_date TEXT,
    customer_id INTEGER,
    customer_name TEXT,
    customer_email TEXT,
    customer_address TEXT,
    products TEXT,
    total_amount REAL
)
''')

# Insert problematic data with redundancy and repeating groups
sample_unf = [
    (1, '2025-01-15', 101, 'John Doe', 'john@email.com', '123 Main St', 'Laptop, Mouse, Keyboard', 1250.00),
    (2, '2025-01-16', 102, 'Jane Smith', 'jane@email.com', '456 Oak Ave', 'Monitor', 300.00),
    (3, '2025-01-17', 101, 'John Doe', 'john@email.com', '123 Main St', 'Headphones', 150.00)
]
cursor.executemany("INSERT INTO orders_unf VALUES (?, ?, ?, ?, ?, ?, ?, ?)", sample_unf)
conn.commit()

cursor.execute("SELECT * FROM orders_unf")
pd.DataFrame(cursor.fetchall(), columns=['order_id', 'order_date', 'customer_id', 'customer_name', 'customer_email', 'customer_address', 'products', 'total_amount'])

=== UNF: Unnormalized Form ===


,order_id,order_date,customer_id,customer_name,customer_email,customer_address,products,total_amount
0,1,2025-01-15,101,John Doe,john@email.com,123 Main St,"Laptop, Mouse, Keyboard",1250.0
1,2,2025-01-16,102,Jane Smith,jane@email.com,456 Oak Ave,Monitor,300.0
2,3,2025-01-17,101,John Doe,john@email.com,123 Main St,Headphones,150.0
3,1,2025-01-15,101,John Doe,john@email.com,123 Main St,"Laptop, Mouse, Keyboard",1250.0
4,2,2025-01-16,102,Jane Smith,jane@email.com,456 Oak Ave,Monitor,300.0
5,3,2025-01-17,101,John Doe,john@email.com,123 Main St,Headphones,150.0
6,1,2025-01-15,101,John Doe,john@email.com,123 Main St,"Laptop, Mouse, Keyboard",1250.0
7,2,2025-01-16,102,Jane Smith,jane@email.com,456 Oak Ave,Monitor,300.0
8,3,2025-01-17,101,John Doe,john@email.com,123 Main St,Headphones,150.0
9,1,2025-01-15,101,John Doe,john@email.com,123 Main St,"Laptop, Mouse, Keyboard",1250.0


In [28]:
# Cell 2: 1NF (First Normal Form)
print("=== 1NF: Atomic Values, No Repeating Groups ===")
cursor.execute('''
CREATE TABLE IF NOT EXISTS orders_1nf (
    order_id INTEGER,
    order_date TEXT,
    customer_id INTEGER,
    customer_name TEXT,
    customer_email TEXT,
    customer_address TEXT,
    product_name TEXT,
    product_price REAL,
    quantity INTEGER,
    PRIMARY KEY (order_id, product_name)
)
''')

# Clear existing data to allow re-running the cell
cursor.execute("DELETE FROM orders_1nf")

# Split repeating groups into atomic rows (still redundant)
sample_1nf = [
    (1, '2025-01-15', 101, 'John Doe', 'john@email.com', '123 Main St', 'Laptop', 1000.00, 1),
    (1, '2025-01-15', 101, 'John Doe', 'john@email.com', '123 Main St', 'Mouse', 50.00, 1),
    (1, '2025-01-15', 101, 'John Doe', 'john@email.com', '123 Main St', 'Keyboard', 200.00, 1),
    (2, '2025-01-16', 102, 'Jane Smith', 'jane@email.com', '456 Oak Ave', 'Monitor', 300.00, 1),
    (3, '2025-01-17', 101, 'John Doe', 'john@email.com', '123 Main St', 'Headphones', 150.00, 1)
]
cursor.executemany("INSERT INTO orders_1nf VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", sample_1nf)
conn.commit()

cursor.execute("SELECT * FROM orders_1nf ORDER BY order_id, product_name")
pd.DataFrame(cursor.fetchall(), columns=['order_id', 'order_date', 'customer_id', 'customer_name', 'customer_email', 'customer_address', 'product_name', 'product_price', 'quantity'])

=== 1NF: Atomic Values, No Repeating Groups ===


,order_id,order_date,customer_id,customer_name,customer_email,customer_address,product_name,product_price,quantity
0,1,2025-01-15,101,John Doe,john@email.com,123 Main St,Keyboard,200.0,1
1,1,2025-01-15,101,John Doe,john@email.com,123 Main St,Laptop,1000.0,1
2,1,2025-01-15,101,John Doe,john@email.com,123 Main St,Mouse,50.0,1
3,2,2025-01-16,102,Jane Smith,jane@email.com,456 Oak Ave,Monitor,300.0,1
4,3,2025-01-17,101,John Doe,john@email.com,123 Main St,Headphones,150.0,1


In [29]:
# Cell 3: 2NF (Second Normal Form)
print("=== 2NF: Remove Partial Dependencies ===")
cursor.execute('''
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    customer_email TEXT,
    customer_address TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS orders_2nf (
    order_id INTEGER PRIMARY KEY,
    order_date TEXT,
    customer_id INTEGER,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS order_items (
    order_id INTEGER,
    product_name TEXT,
    product_price REAL,
    quantity INTEGER,
    PRIMARY KEY (order_id, product_name),
    FOREIGN KEY (order_id) REFERENCES orders_2nf(order_id)
)
''')

# Clear existing data (delete in reverse order of dependencies)
cursor.execute("DELETE FROM order_items")
cursor.execute("DELETE FROM orders_2nf")
cursor.execute("DELETE FROM customers")

# Insert into separate tables
cursor.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", [
    (101, 'John Doe', 'john@email.com', '123 Main St'),
    (102, 'Jane Smith', 'jane@email.com', '456 Oak Ave')
])

cursor.executemany("INSERT INTO orders_2nf VALUES (?, ?, ?)", [
    (1, '2025-01-15', 101),
    (2, '2025-01-16', 102),
    (3, '2025-01-17', 101)
])

cursor.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", [
    (1, 'Laptop', 1000.00, 1),
    (1, 'Mouse', 50.00, 1),
    (1, 'Keyboard', 200.00, 1),
    (2, 'Monitor', 300.00, 1),
    (3, 'Headphones', 150.00, 1)
])
conn.commit()

# Display all 2NF tables
print("Customers:")
cursor.execute("SELECT * FROM customers")
display(pd.DataFrame(cursor.fetchall(), columns=['customer_id', 'customer_name', 'customer_email', 'customer_address']))

print("\nOrders:")
cursor.execute("SELECT * FROM orders_2nf")
display(pd.DataFrame(cursor.fetchall(), columns=['order_id', 'order_date', 'customer_id']))

print("\nOrder Items:")
cursor.execute("SELECT * FROM order_items")
display(pd.DataFrame(cursor.fetchall(), columns=['order_id', 'product_name', 'product_price', 'quantity']))

=== 2NF: Remove Partial Dependencies ===
Customers:


,customer_id,customer_name,customer_email,customer_address
0,101,John Doe,john@email.com,123 Main St
1,102,Jane Smith,jane@email.com,456 Oak Ave



Orders:


,order_id,order_date,customer_id
0,1,2025-01-15,101
1,2,2025-01-16,102
2,3,2025-01-17,101



Order Items:


,order_id,product_name,product_price,quantity
0,1,Laptop,1000.0,1
1,1,Mouse,50.0,1
2,1,Keyboard,200.0,1
3,2,Monitor,300.0,1
4,3,Headphones,150.0,1


In [30]:
# Cell 4: 3NF (Third Normal Form)
print("=== 3NF: Remove Transitive Dependencies ===")
cursor.execute('''
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT UNIQUE,
    category TEXT,
    supplier TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS order_items_3nf (
    order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price REAL,
    FOREIGN KEY (order_id) REFERENCES orders_2nf(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
''')

# Insert product master data (using OR REPLACE to handle duplicates)
cursor.executemany("INSERT OR REPLACE INTO products (product_id, product_name, category, supplier) VALUES (?, ?, ?, ?)", [
    (1, 'Laptop', 'Electronics', 'TechCorp'),
    (2, 'Mouse', 'Accessories', 'GadgetInc'),
    (3, 'Keyboard', 'Accessories', 'GadgetInc'),
    (4, 'Monitor', 'Electronics', 'TechCorp'),
    (5, 'Headphones', 'Audio', 'SoundPro')
])

# Clear order_items_3nf before inserting (since it uses AUTOINCREMENT, we can't use OR REPLACE easily)
cursor.execute("DELETE FROM order_items_3nf")

# Insert into new order items structure
cursor.executemany("INSERT INTO order_items_3nf (order_id, product_id, quantity, unit_price) VALUES (?, ?, ?, ?)", [
    (1, 1, 1, 1000.00),
    (1, 2, 1, 50.00),
    (1, 3, 1, 200.00),
    (2, 4, 1, 300.00),
    (3, 5, 1, 150.00)
])
conn.commit()

print("Products (master data):")
cursor.execute("SELECT * FROM products")
display(pd.DataFrame(cursor.fetchall(), columns=['product_id', 'product_name', 'category', 'supplier']))

print("\nOrder Items (3NF):")
cursor.execute("SELECT * FROM order_items_3nf")
display(pd.DataFrame(cursor.fetchall(), columns=['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']))

=== 3NF: Remove Transitive Dependencies ===
Products (master data):


,product_id,product_name,category,supplier
0,1,Laptop,Electronics,TechCorp
1,2,Mouse,Accessories,GadgetInc
2,3,Keyboard,Accessories,GadgetInc
3,4,Monitor,Electronics,TechCorp
4,5,Headphones,Audio,SoundPro



Order Items (3NF):


,order_item_id,order_id,product_id,quantity,unit_price
0,11,1,1,1,1000.0
1,12,1,2,1,50.0
2,13,1,3,1,200.0
3,14,2,4,1,300.0
4,15,3,5,1,150.0


In [31]:
# Cell 5: Query 3NF Schema
print("=== Query: Reconstruct Data from 3NF ===")
cursor.execute('''
SELECT 
    o.order_id,
    o.order_date,
    c.customer_name,
    p.product_name,
    oi.quantity,
    oi.unit_price,
    (oi.quantity * oi.unit_price) as line_total
FROM orders_2nf o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items_3nf oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
ORDER BY o.order_id, p.product_name
''')

df_result = pd.DataFrame(cursor.fetchall(), columns=['order_id', 'order_date', 'customer_name', 'product_name', 'quantity', 'unit_price', 'line_total'])
print("Normalized data joined together:")
display(df_result)

print("\n=== Normalization Benefits Summary ===")
print(f"UNF → 1NF: Eliminated {cursor.execute('SELECT COUNT(DISTINCT customer_id) FROM orders_unf').fetchone()[0]} duplicate customer records (partial)")
print(f"1NF → 2NF: Created dedicated customer table")
print(f"2NF → 3NF: Created dedicated product table")
print(f"Final tables: customers, orders_2nf, products, order_items_3nf")

=== Query: Reconstruct Data from 3NF ===
Normalized data joined together:


,order_id,order_date,customer_name,product_name,quantity,unit_price,line_total
0,1,2025-01-15,John Doe,Keyboard,1,200.0,200.0
1,1,2025-01-15,John Doe,Laptop,1,1000.0,1000.0
2,1,2025-01-15,John Doe,Mouse,1,50.0,50.0
3,2,2025-01-16,Jane Smith,Monitor,1,300.0,300.0
4,3,2025-01-17,John Doe,Headphones,1,150.0,150.0



=== Normalization Benefits Summary ===
UNF → 1NF: Eliminated 2 duplicate customer records (partial)
1NF → 2NF: Created dedicated customer table
2NF → 3NF: Created dedicated product table
Final tables: customers, orders_2nf, products, order_items_3nf


In [ ]:
conn.close()